# Transcribe


In [ ]:
# @title Part 0: Install dependencies


# 1. Cleanly install torch, pandas, and whisperx together
!pip install --force-reinstall torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 whisperx==3.8.5

# 2. Heal and sync pandas/numpy so Colab's C-extensions don't throw circular import errors
!pip install --upgrade pandas==2.2.3 numpy

import subprocess
subprocess.run([
    "curl",
    "-d", "Google Colab. Finished installing dependencies",
    f"https://ntfy.sh/your_topic_here"
])

# 3. Automatically kill and restart the Python kernel
import os
os.kill(os.getpid(), 9)

In [ ]:
# @title Part 1: Run Transcription & Diarization
import gc
import glob
import os
import subprocess

import nltk
import torch
import whisperx
from google.colab import userdata
from whisperx.diarize import DiarizationPipeline

# --- CONFIGURATION & CONSTANTS ---
HF_TOKEN = userdata.get('HF_TOKEN')
NTFY_TOPIC_URL = "https://ntfy.sh/your_topic_here"  # Replace with your ntfy topic or use userdata
DEVICE = "cuda"
COMPUTE_TYPE = "float16"
BATCH_SIZE = 16
VALID_EXTENSIONS = (".mp4", ".m4a", ".mp3", ".wav", ".mkv", ".webm")


def send_notification(title: str, message: str, topic_url: str = NTFY_TOPIC_URL):
    """
    Sends a push notification via ntfy using curl.
    """
    subprocess.run([
        "curl",
        "-H", f"Title: {title}",
        "-d", message,
        topic_url
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)


def convert_to_mono_ogg(media_path, output_ogg_path):
    """
    Converts any input video or audio file to 32kbps mono OGG using FFmpeg.
    """
    print(f"Converting to 32k Mono OGG: {media_path} -> {output_ogg_path}...")

    command = [
        'ffmpeg', '-y',
        '-i', media_path,
        '-vn',                # Strip video
        '-ac', '1',           # Mono channel
        '-c:a', 'libvorbis',    # OGG Vorbis encoder
        '-b:a', '32k',          # 32kbps bitrate
        '-ar', '16000',         # 16kHz sample rate (Whisper native)
        output_ogg_path
    ]

    try:
        subprocess.run(command, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        print(f"Conversion complete and saved: {output_ogg_path}")
        return output_ogg_path
    except subprocess.CalledProcessError as e:
        print(f"Error converting with FFmpeg: {e}")
        return None


def transcribe_audio(audio_path):
    """
    Transcribes, aligns, and diarizes audio using WhisperX.
    """
    os.environ['NLTK_ALLOW_PROXIED_URLOPEN'] = '1'

    try:
        nltk.download('punkt_tab', raise_on_error=True)
    except Exception as e:
        print(f"Warning downloading punkt_tab: {e}")

    # 1. Transcribe
    model = whisperx.load_model("large-v3", DEVICE, compute_type=COMPUTE_TYPE)
    audio = whisperx.load_audio(audio_path)
    result = model.transcribe(audio, batch_size=BATCH_SIZE)

    # Cleanup Transcribe Model
    del model
    gc.collect()
    torch.cuda.empty_cache()

    # 2. Align
    model_a, metadata = whisperx.load_align_model(language_code=result["language"], device=DEVICE)
    result = whisperx.align(result["segments"], model_a, metadata, audio, DEVICE, return_char_alignments=False)

    del model_a
    gc.collect()
    torch.cuda.empty_cache()

    # 3. Diarize
    diarize_model = DiarizationPipeline(token=HF_TOKEN, device=DEVICE)
    diarize_segments = diarize_model(audio)
    result = whisperx.assign_word_speakers(diarize_segments, result)

    # Clean text formatting
    transcription_body = ""
    last_speaker = None
    for segment in result["segments"]:
        speaker = segment.get("speaker", "UNKNOWN")
        text = segment['text'].strip()
        if speaker != last_speaker:
            transcription_body += f"\n{speaker}: {text} "
            last_speaker = speaker
        else:
            transcription_body += f"{text} "

    return transcription_body.strip()


if __name__ == "__main__":
    transcriptions_completed = 0

    # Gather all media files in directory
    media_files = [
        f for f in os.listdir(".")
        if f.lower().endswith(VALID_EXTENSIONS)
    ]

    for media_file in media_files:
        base_name = os.path.splitext(media_file)[0]
        expected_ogg_file = f"{base_name}.ogg"
        expected_txt_file = f"{base_name}.txt"

        if os.path.exists(expected_txt_file):
            print(f"Skipping {media_file}: {expected_txt_file} already exists.")
            continue

        print(f"\nProcessing file: {media_file}")

        # Reuse existing .ogg if already generated in a previous run
        if os.path.exists(expected_ogg_file):
            print(f"Using existing OGG file: {expected_ogg_file}")
            audio_ogg = expected_ogg_file
        else:
            audio_ogg = convert_to_mono_ogg(media_file, expected_ogg_file)

        if audio_ogg:
            print(f"Transcribing with WhisperX: {audio_ogg}...")
            complete_transcription = transcribe_audio(audio_ogg)

            with open(expected_txt_file, "w", encoding="utf-8") as f:
                f.write(complete_transcription)
            print(f"Transcription saved to: {expected_txt_file}")
            transcriptions_completed += 1

    print(f"\nTranscription process completed. {transcriptions_completed} file(s) processed.")
    send_notification(
        title="Part 1: Transcription Completed",
        message=f"Transcription process completed. {transcriptions_completed} file(s) processed."
    )

In [ ]:
# @title Part 2: Generate Minutes with Gemini
import glob
import os
import subprocess
import time

import requests
from google import genai
from google.colab import userdata
from google.genai import types

# --- CONFIGURATION & CONSTANTS ---
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NTFY_TOPIC_URL = "https://ntfy.sh/your_topic_here"  # Replace with your ntfy topic or use userdata
GEMINI_MODELS = [
    "gemini-3.8-flash",
    "gemini-3.7-flash"
]
SKILL_URL = "https://raw.githubusercontent.com/alirezarezvani/claude-skills/main/project-management/skills/meeting-analyzer/SKILL.md"
CONTEXT_FILE = "context.md"


def send_notification(title: str, message: str, topic_url: str = NTFY_TOPIC_URL):
    """
    Sends a push notification via ntfy using curl.
    """
    subprocess.run([
        "curl",
        "-H", f"Title: {title}",
        "-d", message,
        topic_url
    ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)


def get_skill_instructions():
    """Downloads SKILL.md from GitHub."""
    print("Downloading 'meeting-analyzer' skill from GitHub...")
    try:
        response = requests.get(SKILL_URL, timeout=10)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Error downloading SKILL.md: {e}")
        return "You are an expert project manager and meeting analysis assistant."


def load_custom_context():
    """Loads additional custom context/domain rules from context.md if available."""
    if os.path.exists(CONTEXT_FILE):
        try:
            with open(CONTEXT_FILE, "r", encoding="utf-8") as f:
                print(f"Loaded custom context from {CONTEXT_FILE}.")
                return f.read().strip()
        except Exception as e:
            print(f"Error reading {CONTEXT_FILE}: {e}")
    print(f"No {CONTEXT_FILE} found or error reading it. Proceeding without custom context.")
    return ""


def generate_gemini_minutes(transcript_text, base_name, skill_instructions, custom_context):
    """Sends transcription text to Gemini API to format meeting minutes."""
    client = genai.Client(api_key=GEMINI_API_KEY)

    print(f"\nSending '{base_name}' to Gemini for minutes generation...")

    retry_delay_seconds = 5

    additional_context = f"""{custom_context}

Generate a detailed meeting minute report based on the attached transcript. The minutes consist of three main sections: Meeting Details, Topics Discussed, and a Communication Analysis. For the first two sections, adhere strictly to the following Markdown structure and formatting without omitting any section.

### Structural Instructions:

1. **Main Title:** Use `# Meeting Minutes: [Main Topic Title or Project Name]`.
2. **Initial Metadata:**
    - **Date:** YYYY-MM-DD (Deduce this from the following filename: {base_name})
    - **Context:** Project coordination meeting
    - **Participants:** List identified attendees. Map transcription speaker labels to names and roles where deducible (e.g., `**Name** - Role/Specialty (SPEAKER_XX)`).
3. **Main Body (Topic-by-Topic Development):**
    - Divide the meeting into independent topics using level-2 headers (`## Topic X: [Topic Name]`).
    - Each topic must strictly contain two subsections:
        - `### Overview`: A concise summary of the problem, context, or decision for that topic.
        - `### Questions and Answers`: Bulleted list with raised questions in bold, followed by a sub-bullet for the given answer (e.g., `**[Question]?**\n * **Answer:** [Detailed answer]`).
4. **Closing (Pending Items and Action Items):**
    - Create a final section `## Pending Items and Action Items`.
    - List assigned tasks indicating the action item, responsible party, and deadline or milestone if mentioned in the text.

### Style Rules:

- Maintain a formal, technical, and concise tone.
- Use bold text for participant names and question headers.
- Respect indentation for answers within the question lists to keep the Markdown layout clean.
"""

    user_prompt = f"{additional_context}\n\n=== TRANSCRIPT ===\n{transcript_text}"

    for current_model in GEMINI_MODELS:
        try:
            print(f"Trying model: {current_model}...")
            config = types.GenerateContentConfig(
                system_instruction=skill_instructions,
                temperature=0.3,
            )

            response = client.models.generate_content(
                model=current_model,
                contents=user_prompt,
                config=config
            )

            model_version = current_model.split("-")[1]
            markdown_file = f"{base_name}_MINUTES_{model_version}.md"

            with open(markdown_file, "w", encoding="utf-8") as f:
                f.write(response.text)
                f.write("\n\n---\n")
                f.write(f"> *This document was assisted by artificial intelligence ({current_model}). Please verify accuracy of critical details.*\n")

            print(f"Success! Minutes saved to: {markdown_file} using {current_model}")
            send_notification(
                title="Minutes Completed",
                message=f"Success! Minutes saved to: {markdown_file} using {current_model}"
            )
            return True

        except Exception as e:
            print(f"Failed with {current_model}. Error: {e}")
            if current_model != GEMINI_MODELS[-1]:
                print(f"Downgrading to previous version in {retry_delay_seconds} seconds...\n")
                time.sleep(retry_delay_seconds)
            else:
                print("All available models in the fallback list have been exhausted.")

    send_notification(
        title="Critical Error Generating Minutes",
        message=f"Generation failed for {base_name} after trying all available models."
    )
    return False


if __name__ == "__main__":
    txt_files = glob.glob("*.txt")
    skill_instructions = get_skill_instructions()
    custom_context = load_custom_context()
    minutes_generated = 0

    if not txt_files:
        print("No .txt files found to process. Run Part 1 first.")
    else:
        print(f"Detected {len(txt_files)} .txt file(s)...")
        for txt_file in txt_files:
            base_name = os.path.splitext(txt_file)[0]

            existing_minutes = glob.glob(f"{base_name}_MINUTES*.md")

            if existing_minutes:
                print(f"Skipping {txt_file}: Minutes already exist ({existing_minutes[0]})")
                continue

            with open(txt_file, 'r', encoding='utf-8') as f:
                transcript = f.read().strip()

            if transcript:
                success = generate_gemini_minutes(transcript, base_name, skill_instructions, custom_context)
                if success:
                    minutes_generated += 1
            else:
                print(f"File {txt_file} is empty. Skipping.")

    send_notification(
        title="Part 2: Process Completed",
        message=f"Processed {minutes_generated} new meeting minutes in Google Colab."
    )

In [ ]:
# @title Part 3: Download (.txt, .mp3, and .md)
from google.colab import files

# Compress .txt, .mp3, and .md files without folder structure (-j) and delete originals after zipping (-m)
!cd /content && zip -jm processed_files.zip *.txt *.ogg *.md -x context.md

!rm *.mp4

# Download the zip file to your local computer
files.download('/content/processed_files.zip')